# 04 · Serving: a live endpoint and a dashboard

Put the trained `review-classifier` behind an always-reachable service. The model is
**mounted from the Artifact** produced in Chapter 2, no bucket paths, no glue.

What this chapter shows:

- **Flyte Apps**: a `FastAPIAppEnvironment` turns a plain FastAPI app into a hosted,
  autoscaling service with its own URL.
- **Artifact mount**: `Parameter(value=ArtifactValue(name='review-classifier'))` pins the
  model and Union injects it as a local file at startup.
- **Load once**: `@env.on_startup` loads the pickle once per replica before serving.
- **Open access**: `requires_auth=False`, so you can `curl` it.

> This notebook deploys, tests, and then **deactivates** the app. Requires Chapter 2 to have
> published the `review-classifier` Artifact first.

In [1]:
import time

import requests
import flyte
import flyte.io
import flyte.remote
from fastapi import FastAPI
from flyte.app import ArtifactValue, Parameter
from flyte.app.extras import FastAPIAppEnvironment

# connect to the cluster
flyte.init_from_config('.flyte/config.yaml')

image = flyte.Image.from_debian_base().with_pip_packages('fastapi', 'uvicorn', 'scikit-learn')

## Step 1 · Define the FastAPI app

Routes read the model from a module-level `_state` dict that the startup hook fills.

In [2]:
app = FastAPI(title='Review Radar API', version='1.0.0')
_state = {}

@app.get('/health')
async def health() -> dict:
    return {'status': 'ok', 'model_loaded': 'model' in _state}

# the prediction endpoint the app will serve
@app.get('/predict')
async def predict(text: str) -> dict:
    model = _state.get('model')
    if model is None:
        return {'error': 'model not loaded yet'}
    label = str(model.predict([text])[0])
    confidence = float(model.predict_proba([text]).max())
    return {'text': text, 'predicted': label, 'confidence': confidence}

## Step 2 · Wrap it in an App that mounts the model Artifact

In [3]:
app_env = FastAPIAppEnvironment(
    name='review-radar-api',
    app=app,
    image=image,
    resources=flyte.Resources(cpu=1, memory='1Gi'),
    requires_auth=False,
    scaling=flyte.app.Scaling(replicas=(1, 2)),
    parameters=[
        # mount the model Artifact by name; Union injects it as a local file
        Parameter(name='model', value=ArtifactValue(name='review-classifier', type='file'), download=True),
    ],
)

# load the model once per replica, before the server takes traffic
@app_env.on_startup
async def startup(model: flyte.io.File) -> None:
    import pickle
    with open(model.path, 'rb') as f:
        _state['model'] = pickle.load(f)
    print(f'loaded review-classifier from {model.path}')

## Step 3 · Deploy it

`flyte.serve` builds the image, mounts the Artifact, and starts the service. In a script you
could equivalently run `flyte serve 04_app_api.py app_env` from the `workflows/` folder.

In [4]:
# build the image, mount the Artifact, and start the service
served = flyte.serve(app_env)
url = flyte.remote.App.get(name='review-radar-api').endpoint
print('App URL:', url)

> Building 1 image...

> Building image flyte for environment review-radar-api

i Image 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-107ba66281b44bd5b6ade595933f028d already exists, skipping build

✓ Built image for environment review-radar-api: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-107ba66281b44bd5b6ade595933f028d

VERSION: 811aa52a27bc03b4dc856612a2d47aa3


08:33:34.393492 WARNING   _app_environment.py:334 - Failed to extract app resolver args: Could not find variable name for FastAPI app in module. Skipping       
                         resolver args.

> Deploying app review-radar-api, with image 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-107ba66281b44bd5b6ade595933f028d version 811aa52a27bc03b4dc856612a2d47aa3

08:33:34.628599 WARNING   _app.py:502 - App review-radar-api already exists, updating...

08:33:34.828798 WARNING   _serve.py:727 - Deployed App review-radar-api, you can check the console at                                                           
                         https://demo.hosted.unionai.cloud/v2/domain/development/project/leon-demo/apps/review-radar-api

App URL: https://dawn-mouse-8d69e.apps.demo.hosted.unionai.cloud


## Step 4 · Call the live endpoint

In [5]:
# Wait for the endpoint to answer, then classify a couple of reviews.
# wait for the endpoint to come up, then classify a couple of reviews
deadline = time.time() + 300
while time.time() < deadline:
    try:
        if requests.get(f'{url}/health', timeout=5).json().get('model_loaded'):
            break
    except Exception:
        pass
    time.sleep(10)

print('health :', requests.get(f'{url}/health', timeout=10).json())
print('positive:', requests.get(f'{url}/predict', params={'text': 'this product is amazing i love it'}, timeout=10).json())
print('negative:', requests.get(f'{url}/predict', params={'text': 'broke immediately waste of money'}, timeout=10).json())

health : {'status': 'ok', 'model_loaded': True}
positive: {'text': 'this product is amazing i love it', 'predicted': 'positive', 'confidence': 0.5415873902355615}
negative: {'text': 'broke immediately waste of money', 'predicted': 'negative', 'confidence': 0.6418067017592065}


## Step 5 · Deactivate the app

Apps keep serving (and can hold a warm replica) until you stop them.

In [6]:
# stop the app so it doesn't keep a replica warm
flyte.remote.App.get(name='review-radar-api').deactivate()
print('review-radar-api deactivated')

review-radar-api deactivated


## Also: an interactive dashboard

The same model can back a **Streamlit** dashboard: any web server works behind an
`AppEnvironment` via `@env.server`. It mounts the identical `review-classifier` Artifact.
Deploy it from the `workflows/` folder:

```bash
flyte serve 04_app_dashboard.py app_env   # prints the URL; open it in a browser
```

See `workflows/04_app_dashboard.py` for the full script.

## Recap

- A trained-model Artifact became a live REST endpoint with a few lines of `AppEnvironment`.
- The model was pinned and injected straight from the Artifact: the serving version is explicit.
- The same pattern serves dashboards, LLM backends, or any web app; all scale to zero when idle.

That closes the loop: **dataset -> model -> predictions -> live service**, every hand-off a
versioned Artifact, every failure recoverable.